# Smart Reply Model — Training Pipeline

**Seq2Seq LSTM** model for generating smart reply suggestions in a chat application.

**What this notebook does:**
1. Downloads the Cornell Movie Dialogs Corpus
2. Preprocesses conversation pairs (context → reply)
3. Builds a vocabulary
4. Defines Encoder-Decoder Seq2Seq architecture (matches `ai/smart_reply/model/seq2seq.py`)
5. Trains the model with teacher forcing
6. Evaluates with sample predictions + BLEU score
7. Exports `model.pt` and `vocab.json` for backend inference

**Runtime:** Use GPU if available (Kaggle / Colab). Falls back to CPU automatically.

In [ ]:
# ============================================================
# Cell 1 — Install dependencies (run once)
# ============================================================
%pip install -q torch numpy matplotlib

In [ ]:
# ============================================================
# Cell 2 — Imports & Device Setup
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import re
import os
import random
import zipfile
import urllib.request
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

# Auto-detect GPU / CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Dataset — Cornell Movie Dialogs Corpus

We download the corpus and extract conversational pairs.  
Each pair: **(context message, reply message)**.

In [ ]:
# ============================================================
# Cell 3 — Download & extract dataset
# ============================================================
DATA_URL = "http://www.cs.cornell.edu/~cristian/data/cornell_movie_dialogs_corpus.zip"
DATA_DIR = "cornell_data"

if not os.path.exists(DATA_DIR):
    print("Downloading Cornell Movie Dialogs Corpus...")
    urllib.request.urlretrieve(DATA_URL, "cornell.zip")
    with zipfile.ZipFile("cornell.zip", "r") as z:
        z.extractall(DATA_DIR)
    print("Done.")
else:
    print("Dataset already downloaded.")

# Find the extracted folder
corpus_dir = None
for root, dirs, files in os.walk(DATA_DIR):
    if "movie_lines.txt" in files:
        corpus_dir = root
        break
print(f"Corpus directory: {corpus_dir}")

In [ ]:
# ============================================================
# Cell 4 — Parse movie lines & conversations
# ============================================================
def load_lines(filename):
    lines = {}
    with open(filename, "r", encoding="iso-8859-1") as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            if len(parts) == 5:
                lines[parts[0]] = parts[4]
    return lines

def load_conversations(filename):
    conversations = []
    with open(filename, "r", encoding="iso-8859-1") as f:
        for line in f:
            parts = line.strip().split(" +++$+++ ")
            if len(parts) == 4:
                line_ids = parts[3].replace("[", "").replace("]", "").replace("'", "").replace(" ", "")
                conversations.append(line_ids.split(","))
    return conversations

lines = load_lines(os.path.join(corpus_dir, "movie_lines.txt"))
conversations = load_conversations(os.path.join(corpus_dir, "movie_conversations.txt"))

print(f"Total lines: {len(lines)}")
print(f"Total conversations: {len(conversations)}")

In [ ]:
# ============================================================
# Cell 5 — Build (context, reply) pairs
# ============================================================
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9\s'?!.,]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

pairs = []
for conv in conversations:
    for i in range(len(conv) - 1):
        inp = lines.get(conv[i], "")
        out = lines.get(conv[i + 1], "")
        if inp and out:
            inp = clean_text(inp)
            out = clean_text(out)
            # Filter by length (keep short-medium messages for chat-like replies)
            if 1 <= len(inp.split()) <= 25 and 1 <= len(out.split()) <= 25:
                pairs.append((inp, out))

random.seed(42)
random.shuffle(pairs)
print(f"Total pairs after filtering: {len(pairs)}")
print(f"\nExample pairs:")
for p in pairs[:5]:
    print(f"  Context: {p[0]}")
    print(f"  Reply  : {p[1]}\n")

## 2. Vocabulary

Matches `ai/smart_reply/utils/vocabulary.py` exactly so exported `vocab.json` works with the backend.

In [ ]:
# ============================================================
# Cell 6 — Vocabulary class (matches ai/smart_reply/utils/vocabulary.py)
# ============================================================
class Vocabulary:
    PAD_TOKEN = "<PAD>"
    SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"
    UNK_TOKEN = "<UNK>"

    def __init__(self, max_vocab_size=10000):
        self.word2idx = {self.PAD_TOKEN: 0, self.SOS_TOKEN: 1, self.EOS_TOKEN: 2, self.UNK_TOKEN: 3}
        self.idx2word = {0: self.PAD_TOKEN, 1: self.SOS_TOKEN, 2: self.EOS_TOKEN, 3: self.UNK_TOKEN}
        self.word_count = Counter()
        self.max_vocab_size = max_vocab_size

    def build_from_texts(self, texts):
        for text in texts:
            tokens = self.tokenize(text)
            self.word_count.update(tokens)
        most_common = self.word_count.most_common(self.max_vocab_size - 4)
        for word, _ in most_common:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    @staticmethod
    def tokenize(text):
        text = text.lower().strip()
        text = re.sub(r"[^a-zA-Z0-9\s'?!.,]", "", text)
        return text.split()

    def encode(self, text, max_len=30):
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(t, self.word2idx[self.UNK_TOKEN]) for t in tokens[:max_len - 2]]
        ids = [self.word2idx[self.SOS_TOKEN]] + ids + [self.word2idx[self.EOS_TOKEN]]
        ids += [self.word2idx[self.PAD_TOKEN]] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        words = []
        for idx in ids:
            word = self.idx2word.get(idx, self.UNK_TOKEN)
            if word == self.EOS_TOKEN:
                break
            if word not in (self.PAD_TOKEN, self.SOS_TOKEN):
                words.append(word)
        return " ".join(words)

    def save(self, path):
        data = {"word2idx": self.word2idx, "idx2word": {str(k): v for k, v in self.idx2word.items()}}
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(data, f)

    def load(self, path):
        with open(path, "r") as f:
            data = json.load(f)
        self.word2idx = data["word2idx"]
        self.idx2word = {int(k): v for k, v in data["idx2word"].items()}

    def __len__(self):
        return len(self.word2idx)

In [ ]:
# ============================================================
# Cell 7 — Build vocabulary from all pairs
# ============================================================
MAX_VOCAB = 10000
MAX_LEN = 30           # max tokens per sentence

all_texts = [p[0] for p in pairs] + [p[1] for p in pairs]
vocab = Vocabulary(max_vocab_size=MAX_VOCAB)
vocab.build_from_texts(all_texts)
print(f"Vocabulary size: {len(vocab)}")

## 3. Dataset & DataLoader

In [ ]:
# ============================================================
# Cell 8 — PyTorch Dataset
# ============================================================
class ChatPairDataset(Dataset):
    def __init__(self, pairs, vocab, max_len):
        self.pairs = pairs
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, trg = self.pairs[idx]
        src_ids = self.vocab.encode(src, self.max_len)
        trg_ids = self.vocab.encode(trg, self.max_len)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(trg_ids, dtype=torch.long)

# Train / val split
split = int(0.9 * len(pairs))
train_pairs = pairs[:split]
val_pairs = pairs[split:]

BATCH_SIZE = 128

train_dataset = ChatPairDataset(train_pairs, vocab, MAX_LEN)
val_dataset = ChatPairDataset(val_pairs, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

## 4. Model Architecture

**Encoder-Decoder Seq2Seq with LSTM** — matches `ai/smart_reply/model/seq2seq.py` exactly.

In [ ]:
# ============================================================
# Cell 9 — Model (mirrors ai/smart_reply/model/seq2seq.py)
# ============================================================
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, hidden, cell):
        embedded = self.dropout(self.embedding(x.unsqueeze(1)))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class Seq2SeqModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim, num_layers, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim, num_layers, dropout)
        self.vocab_size = vocab_size

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        outputs = torch.zeros(batch_size, trg_len, self.vocab_size).to(src.device)
        _, hidden, cell = self.encoder(src)
        input_token = trg[:, 0]  # <SOS>
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = output
            if random.random() < teacher_forcing_ratio:
                input_token = trg[:, t]
            else:
                input_token = output.argmax(1)
        return outputs


# Hyperparameters
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.1
LEARNING_RATE = 1e-3
NUM_EPOCHS = 15

model = Seq2SeqModel(len(vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

## 5. Training Loop

In [ ]:
# ============================================================
# Cell 10 — Training loop
# ============================================================
train_losses = []
val_losses = []
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    # ---- Train ----
    model.train()
    epoch_loss = 0
    tf_ratio = max(0.5, 1.0 - epoch * 0.03)  # anneal teacher forcing

    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio=tf_ratio)
        # output: (batch, trg_len, vocab) -> flatten
        output = output[:, 1:].reshape(-1, output.shape[-1])
        trg = trg[:, 1:].reshape(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # ---- Validate ----
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, trg in val_loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=0)
            output = output[:, 1:].reshape(-1, output.shape[-1])
            trg = trg[:, 1:].reshape(-1)
            val_loss += criterion(output, trg).item()

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_smart_reply_model.pt")

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | TF={tf_ratio:.2f} | Train Loss={avg_train:.4f} | Val Loss={avg_val:.4f} | LR={lr:.6f}")

print(f"\nBest val loss: {best_val_loss:.4f}")

In [ ]:
# ============================================================
# Cell 11 — Plot training curves
# ============================================================
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Smart Reply — Training Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluation — Sample Predictions & BLEU

In [ ]:
# ============================================================
# Cell 12 — Generate replies for sample inputs
# ============================================================
model.load_state_dict(torch.load("best_smart_reply_model.pt", map_location=device))
model.eval()

def generate_reply(text, temperature=0.8, max_len=30):
    """Generate a single reply from input text."""
    input_ids = torch.tensor([vocab.encode(text, MAX_LEN)]).to(device)
    with torch.no_grad():
        _, hidden, cell = model.encoder(input_ids)
        token = torch.tensor([vocab.word2idx[Vocabulary.SOS_TOKEN]]).to(device)
        generated = []
        for _ in range(max_len):
            output, hidden, cell = model.decoder(token, hidden, cell)
            probs = torch.softmax(output / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1).squeeze(-1)
            tid = next_token.item()
            if tid == vocab.word2idx[Vocabulary.EOS_TOKEN]:
                break
            generated.append(tid)
            token = next_token
    return vocab.decode(generated)

def generate_multiple(text, n=3):
    """Generate n diverse replies using temperature sampling."""
    replies = set()
    for temp in [0.7, 0.9, 1.1, 0.8, 1.0]:
        r = generate_reply(text, temperature=temp)
        if r and r not in replies:
            replies.add(r)
        if len(replies) >= n:
            break
    return list(replies)[:n]

# Test with sample inputs
test_inputs = [
    "how are you doing?",
    "hi there!",
    "what do you think about that?",
    "thanks for your help",
    "see you later",
    "are you free tomorrow?",
    "i had a great day today",
    "that movie was amazing",
]

print("=" * 60)
print("Smart Reply — Sample Predictions")
print("=" * 60)
for inp in test_inputs:
    replies = generate_multiple(inp, n=3)
    print(f"\n  Input : {inp}")
    for i, r in enumerate(replies, 1):
        print(f"  Reply {i}: {r}")

In [ ]:
# ============================================================
# Cell 13 — BLEU Score on validation set
# ============================================================
from collections import Counter

def compute_bleu_ngram(reference, hypothesis, n=4):
    """Simple BLEU-n implementation."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    if len(hyp_tokens) == 0:
        return 0.0
    scores = []
    for i in range(1, n + 1):
        ref_ngrams = Counter([tuple(ref_tokens[j:j+i]) for j in range(len(ref_tokens) - i + 1)])
        hyp_ngrams = Counter([tuple(hyp_tokens[j:j+i]) for j in range(len(hyp_tokens) - i + 1)])
        overlap = sum((hyp_ngrams & ref_ngrams).values())
        total = sum(hyp_ngrams.values())
        scores.append(overlap / total if total > 0 else 0)
    # Geometric mean
    score = 1.0
    for s in scores:
        score *= max(s, 1e-10)
    score = score ** (1.0 / n)
    # Brevity penalty
    bp = min(1.0, len(hyp_tokens) / max(len(ref_tokens), 1))
    return bp * score

# Sample BLEU on first 500 val pairs
bleu_scores = []
for src_text, ref_text in val_pairs[:500]:
    hyp = generate_reply(src_text, temperature=0.8)
    bleu = compute_bleu_ngram(ref_text, hyp, n=2)  # BLEU-2 for short sentences
    bleu_scores.append(bleu)

avg_bleu = np.mean(bleu_scores)
print(f"Average BLEU-2 on 500 val samples: {avg_bleu:.4f}")
print(f"(Note: BLEU is low for generative chat models — diversity matters more than exact match)")

## 7. Export Model for Backend

Saves `model.pt` and `vocab.json` that the FastAPI backend can load from `ai/saved_models/smart_reply/`.

In [ ]:
# ============================================================
# Cell 14 — Save model & vocab for backend inference
# ============================================================
import shutil
import platform

EXPORT_DIR = "/kaggle/working/smart_reply_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save model weights (CPU for portability)
model.cpu()
torch.save(model.state_dict(), os.path.join(EXPORT_DIR, "model.pt"))

# Save vocabulary
vocab.save(os.path.join(EXPORT_DIR, "vocab.json"))

print(f"Exported to '{EXPORT_DIR}/'")
print(f"  model.pt  — {os.path.getsize(os.path.join(EXPORT_DIR, 'model.pt')) / 1e6:.1f} MB")
print(f"  vocab.json — {os.path.getsize(os.path.join(EXPORT_DIR, 'vocab.json')) / 1e6:.1f} MB")
print(f"\nDownload these files from Kaggle Output tab and place them in:")
print(f"  ai/saved_models/smart_reply/")

In [ ]:
# ============================================================
# Cell 15 — Verify exported files
# ============================================================
export_files = os.listdir(EXPORT_DIR)
print("Files in export directory:")
for f in export_files:
    size = os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1e6
    print(f"  {f} — {size:.2f} MB")

print(f"\n--- Kaggle Instructions ---")
print(f"1. Go to the 'Output' tab on the right sidebar")
print(f"2. Download 'model.pt' and 'vocab.json' from smart_reply_export/")
print(f"3. Place them in your project at: ai/saved_models/smart_reply/")
print(f"4. Restart the backend server — it will auto-load the trained model")